# Predictive Paradox - Power Demand Forecasting

Predicting next hour electricity demand for Bangladesh's power grid using historical data, weather and economic indicators.

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import warnings
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')
import os
os.listdir('/kaggle/input')
os.listdir('/kaggle/input/')
os.listdir('/kaggle/input/datasets')

## Loading the Data

In [ ]:
pgcb = pd.read_excel('/kaggle/input/datasets/ayush20081303/data-predictive-paradox/PGCB_date_power_demand.xlsx')
pgcb.head()

In [ ]:
pgcb.info()

In [ ]:
pgcb.describe()

In [ ]:
weather = pd.read_excel('/kaggle/input/datasets/ayush20081303/data-predictive-paradox/weather_data.xlsx',header=None)
weather = weather.iloc[3:]
weather.columns = weather.iloc[0]
weather = weather.iloc[1:]
weather.columns = ['datetime', 'temperature', 'humidity', 'app_temp',
                   'precipitation', 'dew_point', 'soil_temp', 'wind_dir', 'cloud_cover', 'sunshine']
weather.head()

In [ ]:
economic = pd.read_csv('/kaggle/input/datasets/ayush20081303/data-predictive-paradox/economic_full_1.csv')
economic.head()

## EDA

In [ ]:
pgcb['datetime'] = pd.to_datetime(pgcb['datetime'])
pgcb = pgcb.sort_values('datetime').reset_index(drop=True)

plt.figure(figsize=(15,8))
plt.plot(pgcb['datetime'], pgcb['demand_mw'], linewidth=0.5)
plt.title('Power Demand variation with time')
plt.xlabel('Date')
plt.ylabel('Demand (MW)')
plt.show()

In [ ]:
print("Missing data - pgcb:")
print(pgcb.isnull().sum())

In [ ]:
print("Duplicate rows:", pgcb.duplicated('datetime').sum())

In [ ]:
 plt.figure(figsize=(10,5))

plt.hist(pgcb['demand_mw'],bins=100)

plt.title('Distribution of Demand')
plt.xlabel('Demand (MW)')
plt.ylabel('Frequency')

plt.show()

#notice the rather skewed data

In [ ]:
plt.figure(figsize=(6,4))
plt.boxplot(pgcb['demand_mw'])

plt.title("Boxplot of Demand")
plt.ylabel("Demand (MW)")

plt.show()

#fairly lesser values out of box -> IQR removal is safe

In [ ]:
pgcb['demand_mw'].skew() # data is right skewed

In [ ]:
cols = ['temperature', 'humidity', 'app_temp', 'precipitation']
print("Missing data - weather:")
print(weather[cols].isnull().sum())
print("\n")
for col in cols:
    print(f"{col} -> min = {weather[col].min()}, max = {weather[col].max()}")
print("\n")

print("Duplicate rows:", weather.duplicated('datetime').sum())

In [ ]:
#hourly average pattern -> prediction of next hour demand
pgcb['hour'] = pgcb['datetime'].dt.hour
hourly_avg = pgcb.groupby('hour')['demand_mw'].mean()

plt.figure(figsize=(8,4))
hourly_avg.plot(kind='bar', color='steelblue')
plt.title('Average Demand pattern over Day')
plt.xlabel('Hour')
plt.ylabel('Avg Demand (MW)')
plt.tight_layout()
plt.show()

In [ ]:
#month average pattern -> seasonal variations
pgcb['month'] = pgcb['datetime'].dt.month
monthly_avg = pgcb.groupby('month')['demand_mw'].mean()

plt.figure(figsize=(8,4))
monthly_avg.plot(kind='bar', color='coral')
plt.title('Average Demand pattern over months')
plt.xlabel('Month')
plt.ylabel('Avg Demand (MW)')
plt.tight_layout()
plt.show()

In [ ]:
#year average pattern -> economic factors 
pgcb['year'] = pgcb['datetime'].dt.year
yearly_avg = pgcb.groupby('year')['demand_mw'].mean()

plt.figure(figsize=(8,4))
yearly_avg.plot(kind='bar', color='coral')
plt.title('Average Demand pattern over years')
plt.xlabel('Month')
plt.ylabel('Avg Demand (MW)')
plt.tight_layout()
plt.show()

## Data Preprocessing

### Removing Outliers (IQR Method)

In [ ]:
Q1 = pgcb['demand_mw'].quantile(0.25)
Q3 = pgcb['demand_mw'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Lower bound: {lower_bound:.0f} MW")
print(f"Upper bound: {upper_bound:.0f} MW")

pgcb = pgcb[(pgcb['demand_mw'] >= lower_bound) & (pgcb['demand_mw'] <= upper_bound)]
pgcb = pgcb.drop_duplicates('datetime').reset_index(drop=True)

In [ ]:
# Managing half hours


def smooth_half_hourly(df):
    df = df.sort_values('datetime').reset_index(drop=True)
    is_half = df['datetime'].dt.minute == 30
    
    hourly_df = df[df['datetime'].dt.minute == 0].set_index('datetime')['demand_mw']

    def blend(row):
        if not (row['datetime'].minute == 30):
            return row['demand_mw']
      
        recent_hour = row['datetime'].replace(minute=0, second=0, microsecond=0)
        if recent_hour in hourly_df.index:
            return 0.5 * hourly_df[recent_hour] + 0.5 * row['demand_mw']
        return row['demand_mw']

    df['demand_mw'] = df.apply(blend, axis=1)
    return df


In [ ]:
#Missing timestamps
full_range = pd.date_range(
    start=pgcb['datetime'].min(),
    end=pgcb['datetime'].max(),
    freq='H'
)

pgcb = pgcb.set_index('datetime').reindex(full_range)
pgcb.index.name = 'datetime'
pgcb = pgcb.reset_index()

print(f"Missing demand rows: {pgcb['demand_mw'].isna().sum()}")


### Preparing Weather Data

In [ ]:
weather['datetime'] = pd.to_datetime(weather['datetime'])
for col in weather.columns[1:]:
    weather[col] = pd.to_numeric(weather[col], errors='coerce')

weather = weather.drop_duplicates('datetime').reset_index(drop=True)

print(weather.shape)
weather.head(3)

### Preparing Economic Data

In [ ]:
# selecting the related fields
selected = [
    'Population, total',
    'Electric power consumption (kWh per capita)',
    'Access to electricity, urban (% of urban population)',
    'GDP per capita (current US$)'
]

# filter
econ_sub = economic[economic['Indicator Name'].isin(selected)].copy()

print("Found indicators:")
print(econ_sub['Indicator Name'].unique())

# pick year columns
year_cols = [c for c in econ_sub.columns if c.isdigit()]

# melt
econ_long = econ_sub.melt(
    id_vars=['Indicator Name'],
    value_vars=year_cols,
    var_name='year',
    value_name='value'
)

econ_long['year'] = econ_long['year'].astype(int)

# pivot
econ_wide = econ_long.pivot_table(
    index='year',
    columns='Indicator Name',
    values='value',
    aggfunc='first'
).reset_index()

#demand_mw has data after 2015 only
econ_wide = econ_wide[econ_wide['year'] >= 2015]

#renaming for convinience
rename_map = {
    'Population, total': 'econ_pop_total',
    'Electric power consumption (kWh per capita)': 'econ_power_per_capita',
    'Access to electricity, urban (% of urban population)': 'econ_elec_access_urban',
    'GDP per capita (current US$)':'econ_gdp'
}

econ_wide = econ_wide.rename(columns=rename_map)
print("Related factors after renaming : ")
print(econ_wide.columns)

econ_wide.head()



all_years = pd.DataFrame({'year': range(econ_wide['year'].min(), 2026)})
econ_wide = all_years.merge(econ_wide, on='year', how='left').ffill()

print("\nEconomic data after forward fill:")
econ_wide.tail(5)



### Merging Datasets

In [ ]:
df = pgcb[['datetime', 'demand_mw']].merge(
    weather[['datetime', 'temperature', 'humidity', 'app_temp',
                   'precipitation']],
    on='datetime',
    how='left'
)

df['year'] = df['datetime'].dt.year
df = df.merge(econ_wide, on='year', how='left')
df = df.sort_values('datetime').reset_index(drop=True)

print("Shape after merge:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print("Shape after merge:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

## Feature Engineering

In [ ]:
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['weekend'] = (df['weekday'] >= 5).astype(int)

In [ ]:
# lag features - previous demand values
df['lag_24h'] = df['demand_mw'].shift(24)    # demand 24 hours ago
df['lag_48h'] = df['demand_mw'].shift(48)    # demand 48 hours ago
df['lag_168h'] = df['demand_mw'].shift(168)  # demand 1 week ago

In [ ]:
# rolling mean
df['rolling_mean_24h'] = df['demand_mw'].shift(1).rolling(window=24).mean()


In [ ]:
lag_cols = ['lag_24h', 'lag_48h', 'lag_168h', 'rolling_mean_24h']
for col in lag_cols:
    df[col] = df[col].ffill(limit=3)

In [ ]:
# target = next hour demand
df['target'] = df['demand_mw'].shift(-1)

#drop nans
df = df.dropna(subset=[
    'lag_24h','lag_48h','lag_168h','rolling_mean_24h','target'
]).reset_index(drop=True)

print("Final dataset shape:", df.shape)

df.head()


In [ ]:
# correlation heatmap
econ_cols = [
    'econ_pop_total',
    'econ_power_per_capita',
    'econ_elec_access_urban',
    'econ_gdp'
    
]

feature_cols = [
    'hour','weekday','month','weekend',
    'temperature','humidity','app_temp',
    'precipitation',
    'lag_24h','lag_48h','lag_168h','rolling_mean_24h'
] + econ_cols

plt.figure(figsize=(11,7))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Model Training

### Train Test Split

Using data before 2024 for training and 2024 as test set (chronological split to avoid data leakage).

In [ ]:
train = df[df['datetime'].dt.year < 2024].copy()
test  = df[df['datetime'].dt.year == 2024].copy()

print("Training samples:", len(train))
print("Testing samples:", len(test))

In [ ]:
exclude = ['datetime','target']
features = [col for col in df.columns if col not in exclude]

X_train = train[features]
y_train = train['target']

X_test = test[features]
y_test = test['target']

### LightGBM Regressor

In [ ]:
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

model.fit(X_train, y_train)
print("Training done")

## Evaluation

In [ ]:
predictions = model.predict(X_test)

mape = mean_absolute_percentage_error(y_test, predictions) * 100
mae  = mean_absolute_error(y_test, predictions)
print(f"MAPE: {mape:.2f}%")
print(f"MAE:  {mae:.2f} MW")

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(test['datetime'].values[:500], y_test.values[:500], label='Actual', linewidth=1)
plt.plot(test['datetime'].values[:500], predictions[:500], label='Predicted', linewidth=1, alpha=0.8)
plt.title('Actual vs Predicted Demand (first 500 hours of 2023)')
plt.xlabel('Date')
plt.ylabel('Demand (MW)')
plt.legend()
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
fi = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8,5))
fi.plot(kind='barh', color='steelblue')
plt.title('Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Summary

**Model:** LightBGM Regressor

**Test MAPE:** 2.73%  

**Test Set:** All of 2024

**Training Set:** All years from 2015-2023
